# Llama 3.2 LoRA fine-tuning

## Setup and import libraries

In [ ]:
# Optional: install flash-attention only when it matches your CUDA, PyTorch, and Python versions.
# The notebook can run with SDPA attention if flash-attention is not installed.
# !pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.5.8/flash_attn-2.5.8+cu122torch2.3cxx11abiFALSE-cp310-cp310-linux_x86_64.whl

In [ ]:
import gc
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login
from peft import AutoPeftModelForCausalLM, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from trl import SFTTrainer

## Setup parameters

In [ ]:
# Resolve paths from either the repository root or the training/ folder.
repo_root = Path.cwd()
if not (repo_root / "keys" / "keys.json").exists() and repo_root.name == "training":
    repo_root = repo_root.parent

keys_path = repo_root / "keys" / "keys.json"
with keys_path.open("r", encoding="utf-8") as f:
    keys = json.load(f)

login(token=keys["HF_TOKEN"])
wandb.login(key=keys["WANDB_TOKEN"])

In [ ]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
dataset_name = keys["HF_BT_DATASET"]
dataset_split = "train"

# Local training run name and remote Hugging Face repo for the merged model.
version = "llama-3.2-1b-it-ft-lora-bt"
hf_model_repo = f"{keys['HF_USERNAME']}/{version}"
local_output_dir = repo_root / "training" / "llama-tuning" / "models" / version
local_merged_dir = repo_root / "training" / "merged_model" / version

device_map = "auto"

# LoRA parameters used for the BTGenBot-2 fine-tuning run.
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05
target_modules = ["k_proj", "q_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"]

# Keep dataset splitting and trainer behavior reproducible.
set_seed(1234)

## Load the dataset

In [ ]:
dataset = load_dataset(dataset_name, split=dataset_split)
print(f"dataset size: {len(dataset)}")

## Load the tokenizer and prepare the dataset

In [ ]:
# Load the base tokenizer before formatting examples with the model chat template.
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
def create_message_column(row):
    """Convert one instruction/input/output row into a chat-style supervised example."""
    return {
        "messages": [
            {"role": "system", "content": row["instruction"]},
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["output"]},
        ]
    }


def format_dataset_chatml(row):
    """Serialize chat messages exactly as the base model expects during SFT."""
    return {"text": tokenizer.apply_chat_template(row["messages"], add_generation_prompt=False, tokenize=False)}

In [ ]:
dataset_chatml = dataset.map(create_message_column)
dataset_chatml = dataset_chatml.map(format_dataset_chatml)
dataset_chatml = dataset_chatml.train_test_split(test_size=0.05, seed=1234)
dataset_chatml

### Dataset statistics

In [ ]:
def get_text_token_length(entry):
    return len(tokenizer.tokenize(entry["text"]))


# Inspect sequence lengths to choose a safe max_seq_length for SFTTrainer.
text_token_lengths = dataset_chatml["train"].map(lambda x: {"text_token_length": get_text_token_length(x)}, batched=False)
text_lengths = text_token_lengths["text_token_length"]

shortest_text = min(text_lengths)
longest_text = max(text_lengths)
median_text = np.median(text_lengths)
average_text = np.mean(text_lengths)
text_percentiles = np.percentile(text_lengths, [90, 95, 99])

print(f"Shortest sequence: {shortest_text} tokens")
print(f"Longest sequence: {longest_text} tokens")
print(f"Median sequence length: {median_text:.2f} tokens")
print(f"Average sequence length: {average_text:.2f} tokens")
print(f"90th Percentile: {text_percentiles[0]:.2f} tokens")
print(f"95th Percentile: {text_percentiles[1]:.2f} tokens")
print(f"99th Percentile: {text_percentiles[2]:.2f} tokens")

plt.hist(text_lengths, bins=50, color="blue", alpha=0.7, edgecolor="black")
plt.axvline(x=text_percentiles[0], color="orange", linestyle="--", label="90th Percentile")
plt.axvline(x=text_percentiles[1], color="red", linestyle="--", label="95th Percentile")
plt.axvline(x=text_percentiles[2], color="green", linestyle="--", label="99th Percentile")
plt.axvline(x=median_text, color="purple", linestyle="-", label="Median")
plt.title("Text Length Distribution (Train Split)")
plt.xlabel("Token Length")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## LoRA instruction fine-tuning

In [ ]:
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    compute_dtype = torch.float16
    attn_implementation = "sdpa"

print(attn_implementation)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

# Load the base model in 4-bit precision for LoRA fine-tuning.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=compute_dtype,
    trust_remote_code=True,
    device_map=device_map,
    quantization_config=quantization_config,
    # attn_implementation=attn_implementation,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, add_eos_token=True, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model.resize_token_embeddings(len(tokenizer))

In [ ]:
args = TrainingArguments(
    output_dir=str(local_output_dir),
    eval_strategy="steps",
    do_eval=True,
    optim="paged_adamw_32bit",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    log_level="error",
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    learning_rate=1e-4,
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    eval_steps=100,
    num_train_epochs=5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    report_to="wandb",
    seed=42,
    hub_private_repo=True,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)

### Weights & Biases

In [ ]:
# W&B authentication is performed in the setup cell; this section initializes the run.
project_name = "BTGenBot-2-Llama-3.2-1B-LoRA"
wandb.init(project=project_name)

In [ ]:
# Kept for compatibility with older notebook ordering; the W&B run is initialized above.
wandb.run

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_chatml["train"],
    eval_dataset=dataset_chatml["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1180,
    tokenizer=tokenizer,
    args=args,
)

In [ ]:
trainer.train()
trainer.save_model()

In [ ]:
del model
del trainer

gc.collect()
torch.cuda.empty_cache()
gc.collect()

In [ ]:
new_model = AutoPeftModelForCausalLM.from_pretrained(
    args.output_dir,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=compute_dtype,
    trust_remote_code=True,
    device_map=device_map,
)

# Merge LoRA weights into the base model and save a standalone checkpoint locally.
merged_model = new_model.merge_and_unload()
merged_model.save_pretrained(local_merged_dir, trust_remote_code=True, safe_serialization=True)
tokenizer.save_pretrained(local_merged_dir)

In [ ]:
# Push the merged model and tokenizer to the configured Hugging Face repository.
merged_model.push_to_hub(hf_model_repo, private=True)
tokenizer.push_to_hub(hf_model_repo, private=True)